# climagrid: multi-factor stress forecasting (Kaggle)

Trains and evaluates a forecaster for **several stress factors at once**, and
for **each factor independently** picks the training-history length the data
prefers (10 / 15 / 25 years), benchmarks against persistence and climatology,
calibrates the intervals, and saves one model per factor.

> **Why per factor.** "10 years is enough" was measured for thermal aging and
> does not automatically transfer. Rare-event factors (e.g. ice loading) may
> need more history; smooth rolling factors (heat hours, freeze-thaw, soil)
> are persistence-dominated and history barely matters. We let each factor's
> backtest choose, and recommend the model only where it beats the baselines.

> Forecasts are of environmental **stress**, not equipment failure.

## Setup
Enable **Internet** on Kaggle (Settings -> Internet). The forecasting code is
on a feature branch until released:

In [ ]:
%pip install -q "climagrid[ml] @ git+https://github.com/TemidireAdesiji/climagrid@feat/forecasting"

In [ ]:
import json
import os
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

import climagrid
from climagrid.forecasting import ForecastConfig, evaluate
from climagrid.forecasting.backtest import history_ablation
from climagrid.forecasting.dataset import build_supervised_frame, build_training_panel
from climagrid.forecasting.models import LightGBMForecaster

warnings.filterwarnings("ignore")
ON_KAGGLE = Path("/kaggle/working").exists()
OUT = Path("/kaggle/working") if ON_KAGGLE else Path("climagrid_out")
CACHE = OUT / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
print("climagrid", climagrid.__version__, "| output dir:", OUT)

## 1. Assets and configuration

`TARGETS` are the forecastable stress factors (wildfire is excluded: WFIGS has
no historical archive to train on). Trim `TARGETS` or `ABLATION_WINDOWS` to
cut runtime.

In [ ]:
ASSETS_CSV = "your_assets.csv"  # <-- your CSV (asset_id, lat, lon)
if not os.path.exists(ASSETS_CSV):
    local = Path(climagrid.__file__).resolve().parents[2] / "examples" / "data" / "sample_assets.csv"
    if local.exists():
        ASSETS_CSV = str(local)
    else:
        url = "https://raw.githubusercontent.com/TemidireAdesiji/climagrid/main/examples/data/sample_assets.csv"
        ASSETS_CSV = str(OUT / "sample_assets.csv")
        pd.read_csv(url).to_csv(ASSETS_CSV, index=False)
print(len(pd.read_csv(ASSETS_CSV)), "assets from", ASSETS_CSV)

In [ ]:
TARGETS = [
    "feat_thermal_aging_factor",   # IEEE C57.91, instantaneous - strong ML candidate
    "feat_conductor_sag_index",    # IEEE 738, instantaneous - strong
    "feat_ice_loading_risk",       # ASCE 7-22, rare events - may want more history
    "feat_heat_hours_above_35c",   # 168h rolling - persistence-dominated
    "feat_freeze_thaw_cycles",     # 720h rolling - persistence-dominated
    # "feat_soil_saturation_index",  # dropped: NASA-only precip proxy is not
    #   comparable to NRCS-based soil moisture; revisit with a water-balance model
]
ABLATION_WINDOWS = [10, 15, 25]
N_SPLITS, TEST_SIZE = 3, 90

config = ForecastConfig(
    targets=TARGETS, horizon_days=7,
    lags=[1, 2, 3, 7, 14, 30], rolling_windows=[7, 30], quantiles=[0.1, 0.5, 0.9],
    calibrate_intervals=True,   # mondrian (per-season) by default
    cache_dir=CACHE,
)
START_FULL = datetime(2001, 1, 1, tzinfo=timezone.utc)
END = datetime(2025, 12, 31, tzinfo=timezone.utc)

## 2. Build the multi-factor panel once (cached)
Fetches the full history once and computes every factor's daily series. This
is the slow, cached step.

In [ ]:
panel = build_training_panel(ASSETS_CSV, START_FULL, END, config)
if panel.empty:
    raise RuntimeError(
        "Empty panel: no data fetched. On Kaggle enable Internet (Settings -> Internet)."
    )
panel_path = OUT / "daily_panel_full.parquet"
panel.to_parquet(panel_path, index=False)
present = [t for t in TARGETS if t in panel.columns]
print("panel:", panel.shape, "| factors present:", len(present))
panel.head()

## 3. Per-factor history ablation

`history_ablation` reruns the rolling-origin backtest at each history window
for **every** factor. We then pick, per factor, the window with the best mean
skill versus persistence. Expect thermal/conductor_sag near 10 years, ice
loading possibly longer, and the rolling factors roughly indifferent.

This is the heaviest cell (many model fits). Reduce `TARGETS`/`ABLATION_WINDOWS`
if it is too slow.

In [ ]:
ablation = history_ablation(
    panel, config, windows_years=ABLATION_WINDOWS, n_splits=N_SPLITS, test_size_days=TEST_SIZE
)
ablation.to_csv(OUT / "ablation.csv", index=False)

# Pick the best window per factor by mean skill vs persistence.
by_tw = ablation.groupby(["target", "history_years"])["skill_vs_persistence"].mean().reset_index()
chosen = {}
for tgt, g in by_tw.groupby("target"):
    usable = g.dropna(subset=["skill_vs_persistence"])
    if usable.empty:
        # No usable skill (e.g. a near-constant rare-event factor): default to the
        # longest window and let the recommendation fall back to the baseline.
        chosen[tgt] = int(g["history_years"].max())
    else:
        chosen[tgt] = int(usable.loc[usable["skill_vs_persistence"].idxmax(), "history_years"])
print("chosen history window per factor:")
for t in present:
    print(f"  {t:30s} {chosen.get(t)} yr")

## 4. Train, calibrate and save one model per factor

Each factor is trained at its chosen window and its intervals are calibrated
on a held-out final year. We also record whether the model beat persistence
(deploy the model) or not (use the baseline).

In [ ]:
def subset_last_years(frame, years):
    return frame[frame["date"] >= frame["date"].max() - pd.DateOffset(years=years)]

summary_rows = []
for tgt in present:
    yrs = chosen[tgt]
    sub = subset_last_years(panel, yrs)
    sup = build_supervised_frame(sub, tgt, config)
    dts = sorted(sup["date"].unique())
    cstart = pd.Timestamp(dts[-365])
    tcut = cstart - pd.Timedelta(days=config.effective_embargo_days)
    model = (
        LightGBMForecaster(config)
        .fit(sup[sup["date"] < tcut], tgt)
        .calibrate(sup[sup["date"] >= cstart], tgt)
    )
    model.save(OUT / f"model_{tgt}.joblib")
    # Backtest metrics at the chosen window (skill is calibration-independent).
    rows = ablation[(ablation.target == tgt) & (ablation.history_years == yrs)]
    s_pers = rows["skill_vs_persistence"].mean()
    summary_rows.append({
        "factor": tgt, "history_years": yrs,
        "skill_vs_persistence": round(s_pers, 3),
        "skill_vs_climatology": round(rows["skill_vs_climatology"].mean(), 3),
        "raw_interval_coverage": round(rows["interval_coverage"].mean(), 3),
        "recommendation": "lightgbm" if s_pers > 0.02 else "persistence",
    })
    print(f"  saved model_{tgt}.joblib  ({yrs}yr, skill_vs_persistence={s_pers:.3f})")

factor_summary = pd.DataFrame(summary_rows).sort_values("skill_vs_persistence", ascending=False)

## 5. Cross-factor summary and save

Which factors does ML actually help on, at what history length, and how do the
intervals look? `recommendation` is `lightgbm` where the model beats
persistence and `persistence` (use the baseline) where it does not.

In [ ]:
factor_summary

In [ ]:
factor_summary.to_csv(OUT / "factor_summary.csv", index=False)
manifest = {
    "factors": present,
    "panel_file": panel_path.name,
    "horizon_days": config.horizon_days,
    "min_inference_history_days": config.min_inference_history_days,
    "per_factor": {
        r.factor: {
            "history_years": int(r.history_years),
            "model_file": f"model_{r.factor}.joblib",
            "recommendation": r.recommendation,
        }
        for r in factor_summary.itertuples()
    },
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))
print("saved to", OUT, ":")
for p in sorted(OUT.glob("*")):
    if p.is_file():
        print(f"  {p.name:34s} {p.stat().st_size/1024:8.0f} KB")

In [ ]:
# Skill vs persistence by horizon, per factor (at each factor's chosen window).
fig, ax = plt.subplots(figsize=(8, 4))
for tgt in present:
    yrs = chosen[tgt]
    rows = ablation[(ablation.target == tgt) & (ablation.history_years == yrs)]
    by_h = rows.groupby("horizon_day")["skill_vs_persistence"].mean()
    ax.plot(by_h.index, by_h.values, marker="o", label=tgt.replace("feat_", ""))
ax.axhline(0.0, color="grey", ls="--", lw=0.8)
ax.set_xlabel("horizon (days)"); ax.set_ylabel("skill vs persistence")
ax.set_title("ML skill by factor (chosen history window)")
ax.legend(fontsize=7)
fig.tight_layout(); plt.show()

## 6. Inference: forecast every factor from the recent ~30 days
Loads each saved (calibrated) model and forecasts forward from only the recent
history, the production pattern.

In [ ]:
lookback = config.min_inference_history_days + 10
recent = panel.sort_values(["asset_id", "date"]).groupby("asset_id", group_keys=False).tail(lookback)
forecasts = []
for tgt in present:
    model = LightGBMForecaster.load(OUT / f"model_{tgt}.joblib")
    sup_recent = build_supervised_frame(recent, tgt, config)
    latest = sup_recent.sort_values("date").groupby("asset_id", as_index=False).tail(1)
    forecasts.append(model.predict(latest, tgt))
all_forecasts = pd.concat(forecasts, ignore_index=True)
all_forecasts.to_csv(OUT / "forecasts.csv", index=False)
all_forecasts.head(10)

## Takeaways
- Each factor chose its own history window from the data, not an assumption.
- `factor_summary` says where ML earns its keep (`recommendation=lightgbm`)
  and where a baseline is just as good (`persistence`).
- All per-factor models are saved (calibrated) for download, with
  `manifest.json` recording each factor's window and recommendation.
- Still forecasts of environmental stress for inspection lead time, not
  failure prediction.